# IT6017 - Edge AI Smart Camera System
## Phase 1: Model Zoo Standardization & Export Pipeline on Google Colab

This notebook runs on Google Colab (Linux environment with GPU/CPU) to export and standardize the **3x2 Experimental Matrix** (3 architectures * 2 arithmetic precisions) for course IT6017:
1. **SSD MobileNetV2 (300x300)**: FP32 & INT8
2. **EfficientDet-Lite0 (320x320)**: FP32 & INT8
3. **YOLOv8n - Nano (320x320)**: FP32 & INT8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In [ ]:
# Step 1: Install required dependencies
!pip install --upgrade ultralytics ai-edge-litert tensorflow opencv-python

In [ ]:
import os
import urllib.request
from pathlib import Path

output_dir = Path("models")
output_dir.mkdir(exist_ok=True)
print(f"Target directory: {output_dir.resolve()}")

### 1. Export YOLOv8n (320x320) FP32 & INT8 via Ultralytics
Using official Ultralytics weights (`yolov8n.pt`) with static calibration on `coco8.yaml`.

In [ ]:
from ultralytics import YOLO

# Load official YOLOv8n weights
model = YOLO("yolov8n.pt")

# 1. Export FP32 TFLite (320x320)
print("Exporting YOLOv8n FP32...")
fp32_file = model.export(format="tflite", imgsz=320, int8=False)

# 2. Export INT8 TFLite (320x320) with calibration dataset
print("Exporting YOLOv8n INT8 with calibration...")
int8_file = model.export(format="tflite", imgsz=320, int8=True, data="coco8.yaml")

print(f"FP32 exported to: {fp32_file}")
print(f"INT8 exported to: {int8_file}")

### 2. Download Canonical EfficientDet-Lite0 (320x320)
From official Google MediaPipe repository.

In [ ]:
ed_urls = {
    "efficientdet_lite0_fp32.tflite": "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float32/latest/efficientdet_lite0.tflite",
    "efficientdet_lite0_int8.tflite": "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/latest/efficientdet_lite0.tflite",
}

for name, url in ed_urls.items():
    target = output_dir / name
    print(f"Downloading {name}...")
    urllib.request.urlretrieve(url, target)
    print(f"Saved {name} ({target.stat().st_size / (1024*1024):.2f} MB)")

### 3. Download & Standardize SSD MobileNetV2 (300x300)
From Google Coral Edge TPU and TensorFlow Object Detection Model Zoo.

In [ ]:
import tarfile, io, tempfile, tensorflow as tf

# Download official Coral 300x300 INT8 model
coral_url = "https://raw.githubusercontent.com/google-coral/test_data/master/ssd_mobilenet_v2_coco_quant_postprocess.tflite"
urllib.request.urlretrieve(coral_url, output_dir / "ssd_mobilenet_v2_int8.tflite")
print("Downloaded ssd_mobilenet_v2_int8.tflite")

# Convert 300x300 FP32 frozen graph
tf_zoo_url = "http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v2_quantized_300x300_coco_2019_01_03.tar.gz"
with urllib.request.urlopen(tf_zoo_url) as resp:
    archive_bytes = resp.read()

with tempfile.TemporaryDirectory() as tmpdir:
    with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tar:
        tar.extractall(tmpdir)
    pb_path = os.path.join(tmpdir, "ssd_mobilenet_v2_quantized_300x300_coco_2019_01_03", "tflite_graph.pb")
    converter = tf.compat.v1.lite.TFLiteConverter.from_frozen_graph(
        pb_path,
        input_arrays=["normalized_input_image_tensor"],
        output_arrays=["TFLite_Detection_PostProcess", "TFLite_Detection_PostProcess:1", "TFLite_Detection_PostProcess:2", "TFLite_Detection_PostProcess:3"],
        input_shapes={"normalized_input_image_tensor": [1, 300, 300, 3]}
    )
    converter.allow_custom_ops = True
    fp32_bytes = converter.convert()
    with open(output_dir / "ssd_mobilenet_v2_fp32.tflite", "wb") as f:
        f.write(fp32_bytes)
print("Converted and saved ssd_mobilenet_v2_fp32.tflite (300x300)")

### 4. Package All 6 Models for Download
Compress into `models_zoo.zip` to download and place into the local project.

In [ ]:
!zip -r models_zoo.zip models/
print("Done! You can download models_zoo.zip directly from Colab files menu.")